# apertus-eval-prep — ranking stability on Colab

Runtime → Change runtime type → **T4 GPU** (A10 if you have it).

This notebook is the GPU path for the paper in `paper/stability.md`. Mac cannot run vLLM or bitsandbytes.

Start with the **demo** (`stability_smoke.yaml`, `--limit 2`). Then on a laptop run `ci-width` on the downloaded JSON: Wilson CIs are huge on n=4; that is the point. Do **not** uncomment the paper matrix for this claim.

In [ ]:
import os
if os.path.exists("pyproject.toml") and os.path.exists("data/eval_set.jsonl"):
    print("Already in repo root")
elif os.path.exists("apertus-eval-prep/pyproject.toml"):
    %cd apertus-eval-prep
else:
    !git clone https://github.com/Shivani767/apertus-eval-prep.git
    %cd apertus-eval-prep
!pip -q install -e ".[gpu,viz]"

In [ ]:
import torch
assert torch.cuda.is_available(), "Set runtime to GPU (T4) and rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
if not Path("data/official/eval_set.jsonl").exists():
    !pip -q install -e ".[snapshot]"
    !python scripts/snapshot_benchmarks.py
else:
    print("official slices already on disk")

## Demo (minutes)

0.5B, two items per task, a handful of OFAT cells. Proves the pipeline, not a ranking.

In [ ]:
!python -m apertus_eval_prep sweep --config configs/experiments/stability_smoke.yaml \
  --profile t4 --limit 2 --out-dir results/runs --registry results/registry.jsonl

## Full study (hours; resume-safe)

`--profile t4` skips 7B fp16, 7B int8, and 7B vLLM. Re-run this cell after disconnects; completed `config_hash` rows are skipped.

On A10, use `--profile a10` so the 7B fp16 control is included.

In [ ]:
# Uncomment when you want the paper matrix.
# !python -m apertus_eval_prep sweep --config configs/experiments/stability.yaml \
#   --profile t4 --out-dir results/runs --registry results/registry.jsonl
print("Full sweep is commented out. Uncomment the command in this cell.")

In [ ]:
!python -m apertus_eval_prep report --registry results/registry.jsonl --out reports/stability
!python -m apertus_eval_prep paper-tables --registry results/registry.jsonl --out paper/_generated_tables.md
from google.colab import files
from pathlib import Path
for p in Path("results").rglob("*"):
    if p.suffix in {".json", ".jsonl", ".md"} and p.is_file():
        print("download", p)
files.download("results/registry.jsonl") if Path("results/registry.jsonl").exists() else None

Download `results/runs/*.json`, `results/registry.jsonl`, and `reports/stability/` into the git clone and commit them. Do not edit numbers by hand.